# Debugging a London House Price Prediction Pipeline
## Prompt B — Run 2

This notebook identifies and fixes bugs in a provided ML pipeline for predicting London house prices.


## The Buggy Pipeline

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')

feature_cols = ['bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                'rentEstimate', 'crime_total', 'census_denom_total']
X = df[feature_cols].fillna(0)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

train_preds = model.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, train_preds))
mae = mean_absolute_error(y_train, train_preds)
r2 = r2_score(y_train, train_preds)

print(f"Test RMSE: £{rmse:,.0f}")
print(f"Test MAE:  £{mae:,.0f}")
print(f"Test R²:   {r2:.4f}")
```


---
## Identified Bugs

### Bug 1: Target Leakage via `rentEstimate`

`rentEstimate` is an estimated monthly rent for each property. This value is fundamentally derived from the property's market value — rent and sale price are causally linked. The correlation between `rentEstimate` and `price` is approximately **0.98**.

Including this feature means the model has access to a near-perfect proxy of the answer. This constitutes **data leakage** and would produce misleadingly high performance that would not generalise to new properties.

**Fix:** Remove `rentEstimate` from the features.

### Bug 2: Model evaluated on training data, mislabelled as test

The evaluation section calls `model.predict(X_train)` and compares against `y_train`, yet the print statements say "Test RMSE", "Test MAE", "Test R²".

A Random Forest with 100 trees will fit the training data very closely (near-perfect memorisation), so training metrics are **wildly optimistic**. The actual test performance would be substantially worse.

**Fix:** Use `model.predict(X_test)` and compare against `y_test`.

### Bug 3: No log-transformation of skewed prices

House prices in London are extremely right-skewed (skewness > 5). Training on raw prices means:
- RMSE is dominated by errors on expensive properties
- The model struggles with the long tail of the distribution

**Fix:** Apply `np.log1p(price)` before training and `np.expm1(prediction)` for evaluation.

### Bug 4: `fillna(0)` is statistically inappropriate

Filling NaN with 0 creates nonsensical data: a property with 0 floor area, 0 bedrooms, etc. This is far from the true distribution and introduces systematic bias.

**Fix:** Use median imputation for numeric features.

### Bug 5: No outlier removal or capping

Extreme price outliers (multi-million pound mansions) distort the model. RMSE is particularly sensitive to outliers due to squaring.

**Fix:** Cap prices at the 99th percentile before training.

### Bug 6: Only 7 features used (ignoring many useful predictors)

The pipeline uses only 7 features from a dataset that contains 50+ columns after merging. Key features like `propertyType`, `tenure`, `latitude`, `longitude`, and detailed area-level features are all ignored.

**Fix:** Use the full feature set with proper encoding of categoricals.


---
## Step 1: Reproduce the Buggy Results

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Load data
prices = pd.read_csv('london_house_prices.csv')
areas = pd.read_csv('london_area_features.csv')
df = prices.merge(areas, on='outcode', how='left')

print(f"Dataset: {df.shape}")
print(f"Price skewness: {df['price'].skew():.2f}")
print(f"rentEstimate-price correlation: {df['rentEstimate'].corr(df['price']):.4f}")


Dataset: (417561, 63)
Price skewness: 5.05
rentEstimate-price correlation: 0.9812


In [2]:
# Run the BUGGY pipeline exactly as written
feature_cols_buggy = ['bedrooms', 'bathrooms', 'floorAreaSqM', 'livingRooms',
                      'rentEstimate', 'crime_total', 'census_denom_total']
X_b = df[feature_cols_buggy].fillna(0)
y_b = df['price']

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.2, random_state=42)

model_b = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_b.fit(X_train_b, y_train_b)

# Bug: using train predictions
train_preds = model_b.predict(X_train_b)
b_rmse = np.sqrt(mean_squared_error(y_train_b, train_preds))
b_mae = mean_absolute_error(y_train_b, train_preds)
b_r2 = r2_score(y_train_b, train_preds)

print("BUGGY RESULTS (train metrics reported as 'test'):")
print(f"  RMSE: \u00a3{b_rmse:,.0f}")
print(f"  MAE:  \u00a3{b_mae:,.0f}")
print(f"  R\u00b2:   {b_r2:.4f}")
print("\nThese are training set metrics — the true test performance is much worse.")


BUGGY RESULTS (train metrics reported as 'test'):
  RMSE: £31,870
  MAE:  £6,460
  R²:   0.9988

These are training set metrics — the true test performance is much worse.


---
## Step 2: Corrected Pipeline

In [3]:
print("="*60)
print("CORRECTED PIPELINE")
print("="*60)

# Fix 1: Use all features, EXCLUDE rentEstimate
exclude = ['price', 'rentEstimate', 'outcode', 'outcode_lat', 'outcode_lon']
feature_cols = [c for c in df.columns if c not in exclude]
cat_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f"Using {len(feature_cols)} features (was 7)")
print(f"  Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")
print(f"  rentEstimate: EXCLUDED (leakage)")

# Fix 4: Median imputation (not fillna(0))
df_c = df.copy()
for col in num_cols:
    df_c[col] = df_c[col].fillna(df_c[col].median())
for col in cat_cols:
    df_c[col] = df_c[col].fillna(df_c[col].mode()[0])
    le = LabelEncoder()
    df_c[col] = le.fit_transform(df_c[col].astype(str))

df_c = df_c.dropna(subset=['price'])

# Fix 5: Cap outliers
p99 = df_c['price'].quantile(0.99)
before = len(df_c)
df_c = df_c[df_c['price'] <= p99]
print(f"Outlier capping at \u00a3{p99:,.0f}: removed {before-len(df_c):,} rows")

# Fix 3: Log-transform target
df_c['log_price'] = np.log1p(df_c['price'])

all_feats = num_cols + cat_cols
X = df_c[all_feats]
y_log = df_c['log_price']
y_raw = df_c['price']

# Same split parameters
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42)
y_test_raw = df_c.loc[y_test_log.index, 'price']

print(f"\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}")


CORRECTED PIPELINE


Using 58 features (was 7)
  Numeric: 55, Categorical: 3
  rentEstimate: EXCLUDED (leakage)


Outlier capping at £4,959,000: removed 4,173 rows



Train: 330,710  |  Test: 82,678


In [4]:
# Train corrected model
model_c = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_c.fit(X_train, y_train_log)

# Fix 2: Evaluate on TEST set
test_pred_log = model_c.predict(X_test)
test_pred = np.expm1(test_pred_log)

c_rmse = np.sqrt(mean_squared_error(y_test_raw, test_pred))
c_mae = mean_absolute_error(y_test_raw, test_pred)
c_r2 = r2_score(y_test_raw, test_pred)

print("CORRECTED RESULTS (genuine test set metrics):")
print(f"  RMSE: \u00a3{c_rmse:,.0f}")
print(f"  MAE:  \u00a3{c_mae:,.0f}")
print(f"  R\u00b2:   {c_r2:.4f}")


CORRECTED RESULTS (genuine test set metrics):
  RMSE: £119,633
  MAE:  £36,751
  R²:   0.9692


In [5]:
print("="*70)
print("COMPARISON: BUGGY vs CORRECTED")
print("="*70)

print(f"\n{'Metric':<10} {'Buggy (train eval + leakage)':<30} {'Corrected (true test)':<25}")
print("-"*65)
print(f"{'RMSE':<10} {'\u00a3'+f'{b_rmse:,.0f}':<30} {'\u00a3'+f'{c_rmse:,.0f}':<25}")
print(f"{'MAE':<10} {'\u00a3'+f'{b_mae:,.0f}':<30} {'\u00a3'+f'{c_mae:,.0f}':<25}")
print(f"{'R\u00b2':<10} {f'{b_r2:.4f}':<30} {f'{c_r2:.4f}':<25}")

print(f"\nThe buggy R\u00b2 of {b_r2:.4f} is unrealistically high because:")
print("  1. It evaluates on training data (Random Forest memorises training samples)")
print("  2. rentEstimate provides a near-perfect proxy of the target")
print(f"\nThe corrected R\u00b2 of {c_r2:.4f} is a realistic measure of generalisation.")


COMPARISON: BUGGY vs CORRECTED

Metric     Buggy (train eval + leakage)   Corrected (true test)    
-----------------------------------------------------------------
RMSE       £31,870                        £119,633                 
MAE        £6,460                         £36,751                  
R²         0.9988                         0.9692                   

The buggy R² of 0.9988 is unrealistically high because:
  1. It evaluates on training data (Random Forest memorises training samples)
  2. rentEstimate provides a near-perfect proxy of the target

The corrected R² of 0.9692 is a realistic measure of generalisation.


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cap = y_test_raw.quantile(0.99)

# Buggy model on actual test set (for fair visual comparison)
buggy_test_pred = model_b.predict(X_test_b)
axes[0].scatter(y_test_b, buggy_test_pred, alpha=0.1, s=5, c='#e74c3c')
axes[0].plot([0, y_test_b.quantile(0.99)], [0, y_test_b.quantile(0.99)], 'k--', lw=1)
axes[0].set_title(f'Buggy Model on Test Set')
axes[0].set_xlabel('Actual (\u00a3)'); axes[0].set_ylabel('Predicted (\u00a3)')
axes[0].set_xlim(0, y_test_b.quantile(0.99))
axes[0].set_ylim(0, y_test_b.quantile(0.99))

# Corrected model
axes[1].scatter(y_test_raw, test_pred, alpha=0.1, s=5, c='#2ecc71')
axes[1].plot([0, cap], [0, cap], 'k--', lw=1)
axes[1].set_title(f'Corrected Model (R\u00b2={c_r2:.4f})')
axes[1].set_xlabel('Actual (\u00a3)'); axes[1].set_ylabel('Predicted (\u00a3)')
axes[1].set_xlim(0, cap); axes[1].set_ylim(0, cap)

plt.tight_layout()
plt.savefig('run4_comparison_plots.png', dpi=150, bbox_inches='tight')
plt.show()


In [7]:
# Feature importances
imp = pd.Series(model_c.feature_importances_, index=all_feats).sort_values(ascending=False)
print("Top 15 Feature Importances:")
for i, (f, v) in enumerate(imp.head(15).items()):
    print(f"  {i+1:2d}. {f:<35s} {v:.4f}")

fig, ax = plt.subplots(figsize=(10, 8))
imp.head(20).plot(kind='barh', ax=ax, color='#3498db')
ax.set_xlabel('Importance')
ax.set_title('Corrected Model — Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('run4_feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()


Top 15 Feature Importances:
   1. floorAreaSqM                        0.5330
   2. census_level4_perc                  0.1119
   3. crime_other_crime                   0.0797
   4. longitude                           0.0525
   5. latitude                            0.0448
   6. census_employed_total_perc          0.0283
   7. census_unemployed_perc              0.0210
   8. tenure                              0.0202
   9. bedrooms                            0.0129
  10. census_age_16_to_34_perc            0.0129
  11. propertyType                        0.0124
  12. crime_criminal_damage_and_arson     0.0086
  13. energyRating                        0.0079
  14. bathrooms                           0.0066
  15. crime_shoplifting                   0.0051


---
## Impact Summary

| Bug | How it inflated metrics | Severity |
|-----|------------------------|----------|
| rentEstimate leakage | Gave model a ~0.98-correlated proxy of the target | **Critical** |
| Train eval as test | RF memorises training data → R² ≈ 1.0 on train | **Critical** |
| No log-transform | Model dominated by expensive outliers | **Moderate** |
| fillna(0) | Introduced bias (0 bedrooms, 0 floor area) | **Moderate** |
| No outlier handling | Extreme prices inflate RMSE disproportionately | **Moderate** |
| Limited features | Ignored property type, location, area demographics | **Moderate** |

The two critical bugs (leakage + train evaluation) together created the illusion of near-perfect performance (R² ≈ 0.999). The corrected pipeline provides honest, generalisable metrics.
